In [ ]:
def forward(self, x: torch.Tensor) -> torch.Tensor:
    # 1. Passe x pelo stem (camadas iniciais)
    # Entrada: (B, 3, 256, 256) -> Saída: (B, 64, 64, 64) aprox.
    x = self.stem(x)[cite: 2]

    # 2. Passe x por cada bloco residual (o "corpo" da ResNet)
    # Dependendo de CNN_LAYERS (N), ele passará por 1, 2, 3 ou 4 blocos
    for layer in self.res_layers:
        x = layer(x)[cite: 2]

    # 3. Aplique a projeção linear
    # Isso padroniza o número de canais para ENCODER_DIM (512)
    # Saída: (B, 512, H', W')
    x = self.proj(x)[cite: 2]

    # 4. Reordene e achate as dimensões espaciais
    # Atualmente temos (Batch, Canais, Altura, Largura)
    # Precisamos de (Batch, Pixels, Canais) para a Atenção funcionar
    
    # Movemos o canal (512) para a última posição
    x = x.permute(0, 2, 3, 1) # Vira (B, H', W', 512)
    
    # 'Achatamos' H' e W' em uma dimensão só (num_pixels)
    batch_size = x.size(0)
    x = x.reshape(batch_size, -1, x.size(-1)) # Vira (B, num_pixels, 512)

    return x